In [7]:
import sys
sys.path.append("..")

import pandas as pd
from src.features import calcular_surpresa, calcular_ian, calcular_ice

eventos_cpi = pd.read_csv("../data/eventos.csv")
print(eventos_cpi.head())

eventos_cpi = calcular_surpresa(eventos_cpi)
print(eventos_cpi[["data", "actual", "forecast", "surpresa_zscore"]].head())

  indicador        data  actual  forecast  diferenca  surpresa_zscore
0   CPI_EUA  2026-07-14    -0.4      -0.1       -0.3        -3.022518
1   CPI_EUA  2026-06-10     0.5       0.5        0.0         0.000000
2   CPI_EUA  2026-05-12     0.6       0.6        0.0         0.000000
3   CPI_EUA  2026-04-10     0.9       1.0       -0.1        -1.007506
4   CPI_EUA  2026-03-11     0.3       0.3        0.0         0.000000
         data  actual  forecast  surpresa_zscore
0  2026-07-14    -0.4      -0.1        -0.006288
1  2026-06-10     0.5       0.5         0.000000
2  2026-05-12     0.6       0.6         0.000000
3  2026-04-10     0.9       1.0        -0.002096
4  2026-03-11     0.3       0.3         0.000000


In [8]:
eventos_cpi = calcular_ian(eventos_cpi, termos_busca=["CPI", "inflation report"], geo="US")
print(eventos_cpi[["data", "surpresa_zscore", "IAN"]].head())


        data  surpresa_zscore       IAN
0 2023-01-06         0.482068  0.039216
1 2023-02-03         6.958546  0.068627
2 2023-02-09        -0.000524  0.137255
3 2023-03-10         2.221705  0.088235
4 2023-03-10         0.001258  0.088235


In [9]:
eventos_cpi = calcular_ice(
    eventos_cpi,
    termos_otimistas=["abrir empresa", "comprar carro", "promoção passagens"],
    termos_pessimistas=["perder emprego", "inflação alta", "dívida"],
    geo="BR"
)
print(eventos_cpi[["data", "surpresa_zscore", "IAN", "ICE"]])

          data  surpresa_zscore       IAN       ICE
0   2023-01-06         0.482068  0.039216  0.506530
1   2023-02-03         6.958546  0.068627  0.465924
2   2023-02-09        -0.000524  0.137255  0.463839
3   2023-03-10         2.221705  0.088235  0.414196
4   2023-03-10         0.001258  0.088235  0.414196
..         ...              ...       ...       ...
147 2026-07-02        -1.194690  0.352941  0.067343
148 2026-07-14        -0.006288  0.450980  0.093821
149 2026-08-06         0.000000  0.049020  0.098876
150 2026-08-06         0.000000  0.049020  0.098876
151 2026-08-07        -2.263623  0.049020  0.098876

[152 rows x 4 columns]


In [10]:
eventos_cpi.to_csv("../data/eventos_completo.csv", index=False)
print("Salvo com sucesso!")

Salvo com sucesso!


In [11]:
payroll_bruto = pd.read_csv("../data/payroll.csv", sep=";")
payroll_bruto = payroll_bruto.dropna(subset=["Actual", "Forecast"])

payroll_bruto["Actual"] = payroll_bruto["Actual"].str.replace("K", "", regex=False).astype(float)
payroll_bruto["Forecast"] = payroll_bruto["Forecast"].str.replace("K", "", regex=False).astype(float)

data_limpa = payroll_bruto["Release date"].str.split("(").str[0]
data_limpa = data_limpa.str.replace(r"\s+", " ", regex=True).str.strip()

payroll_bruto["data"] = pd.to_datetime(data_limpa, format="%b %d, %Y", errors="coerce")
print(payroll_bruto[["Release date", "data"]].head(10))
print(f"Quantos viraram NaT: {payroll_bruto['data'].isna().sum()}")

          Release date       data
0   Aug 07, 2026 (Jul) 2026-08-07
1   Jul 02, 2026 (Jun) 2026-07-02
2   Jun 05, 2026 (May) 2026-06-05
3   May 08, 2026 (Apr) 2026-05-08
4   Apr 03, 2026 (Mar) 2026-04-03
5   Mar 06, 2026 (Feb) 2026-03-06
6   Feb 11, 2026 (Jan) 2026-02-11
7   Jan 09, 2026 (Dec) 2026-01-09
8   Dec 16, 2025 (Nov) 2025-12-16
10  Nov 20, 2025 (Sep) 2025-11-20
Quantos viraram NaT: 0


In [12]:
payroll_bruto = payroll_bruto[payroll_bruto["data"] >= "2023-01-01"]

payroll_bruto["indicador"] = "Payroll_EUA"
payroll_eventos = payroll_bruto[["indicador", "data", "Actual", "Forecast"]].rename(
    columns={"Actual": "actual", "Forecast": "forecast"}
)
payroll_eventos["data"] = payroll_eventos["data"].dt.strftime("%Y-%m-%d")

print(payroll_eventos)
print(f"\nTotal de linhas: {len(payroll_eventos)}")

      indicador        data  actual  forecast
0   Payroll_EUA  2026-08-07   -23.0      85.0
1   Payroll_EUA  2026-07-02    57.0     114.0
2   Payroll_EUA  2026-06-05   172.0      85.0
3   Payroll_EUA  2026-05-08   115.0      65.0
4   Payroll_EUA  2026-04-03   178.0      65.0
5   Payroll_EUA  2026-03-06   -92.0      58.0
6   Payroll_EUA  2026-02-11   130.0      66.0
7   Payroll_EUA  2026-01-09    50.0      66.0
8   Payroll_EUA  2025-12-16    64.0      51.0
10  Payroll_EUA  2025-11-20   119.0      53.0
11  Payroll_EUA  2025-09-05    22.0      75.0
12  Payroll_EUA  2025-08-01    73.0     106.0
13  Payroll_EUA  2025-07-03   147.0     111.0
14  Payroll_EUA  2025-06-06   139.0     126.0
15  Payroll_EUA  2025-05-02   177.0     138.0
16  Payroll_EUA  2025-04-04   228.0     137.0
17  Payroll_EUA  2025-03-07   151.0     159.0
18  Payroll_EUA  2025-02-07   143.0     169.0
19  Payroll_EUA  2025-01-10   256.0     164.0
20  Payroll_EUA  2024-12-06   227.0     202.0
21  Payroll_EUA  2024-11-01    12.

In [13]:
eventos_principal = pd.read_csv("../data/eventos.csv")
eventos_atualizado = pd.concat([eventos_principal, payroll_eventos], ignore_index=True)
eventos_atualizado = eventos_atualizado.drop_duplicates()
eventos_atualizado.to_csv("../data/eventos.csv", index=False)

print(eventos_atualizado["indicador"].value_counts())

indicador
Payroll_EUA    43
CPI_EUA        39
IPCA_BR        34
Selic_BR       18
Name: count, dtype: int64


In [14]:
import sys
sys.path.append("..")
from src.features import calcular_surpresa, calcular_ian, calcular_ice

payroll_completo = eventos_atualizado[eventos_atualizado["indicador"] == "Payroll_EUA"].copy()

payroll_completo = calcular_surpresa(payroll_completo)
payroll_completo = calcular_ian(payroll_completo, termos_busca=["nonfarm payrolls", "jobs report"], geo="US")
payroll_completo = calcular_ice(
    payroll_completo,
    termos_otimistas=["abrir empresa", "comprar carro", "promoção passagens"],
    termos_pessimistas=["perder emprego", "inflação alta", "dívida"],
    geo="BR"
)

print(payroll_completo[["data", "surpresa_zscore", "IAN", "ICE"]])

         data  surpresa_zscore       IAN       ICE
0  2023-01-06         0.267704  0.010417  0.506530
1  2023-02-03         3.864244  0.072917  0.465924
2  2023-03-10         1.233765  0.187500  0.414196
3  2023-04-07        -0.034918  0.083333  0.320903
4  2023-05-05         0.849668  0.010417  0.344337
5  2023-06-02         1.850647  0.031250  0.304438
6  2023-07-07        -0.186229  0.093750  0.252832
7  2023-08-04        -0.151311  0.031250  0.211572
8  2023-09-01         0.197868  0.000000  0.209646
9  2023-10-06         1.932122  0.166667  0.091262
10 2023-11-03        -0.349179  0.000000  0.073414
11 2023-12-08         0.221146  0.072917 -0.017158
12 2024-01-05         0.535407  0.020833  0.689352
13 2024-02-02         1.932122  0.041667  0.096039
14 2024-03-08         0.896225  0.041667  0.360528
15 2024-04-05         1.059175  0.104167 -0.165287
16 2024-05-03        -0.733275  0.020833 -0.626959
17 2024-06-07         1.047536  0.083333 -0.597865
18 2024-07-05         0.174589 

In [15]:
payroll_completo.to_csv("../data/eventos_payroll_completo.csv", index=False)

In [16]:
import pandas as pd

cpi = pd.read_csv("../data/eventos_completo.csv")
ipca = pd.read_csv("../data/eventos_ipca_completo.csv")
selic = pd.read_csv("../data/eventos_selic_completo.csv")
payroll = pd.read_csv("../data/eventos_payroll_completo.csv")

eventos_todos = pd.concat([cpi, ipca, selic, payroll], ignore_index=True)
eventos_todos = eventos_todos.drop_duplicates()

eventos_todos.to_csv("../data/eventos_todos_completo.csv", index=False)
print(eventos_todos["indicador"].value_counts())

indicador
Payroll_EUA    86
IPCA_BR        68
CPI_EUA        39
Selic_BR       36
Name: count, dtype: int64
